In [17]:
from datasets import load_dataset
dataset=load_dataset("opus_books","en-es")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 93470
    })
})


In [18]:
from transformers import MarianTokenizer, MarianMTModel
model_name="Helsinki-NLP/opus-mt-es-en"
tokenizer=MarianTokenizer.from_pretrained(model_name)
model=MarianMTModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [19]:
max_length=128
def preprocessing(example):
    inputs = example["translation"]["es"]
    targets = example["translation"]["en"]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [20]:
tokenized_dataset=dataset.map(preprocessing)

In [21]:
from transformers import TrainingArguments,Trainer
training_args=TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=5,
    logging_steps=100,
    save_steps=500
)

In [23]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(2000)),
)
trainer.train()

Step,Training Loss
100,1.551306
200,0.959877
300,0.761715
400,0.645838
500,0.675393
600,0.527270
700,0.532780
800,0.469141
900,0.424366
1000,0.448376


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.6390614273071289, metrics={'train_runtime': 338.868, 'train_samples_per_second': 29.51, 'train_steps_per_second': 3.689, 'total_flos': 338983649280000.0, 'train_loss': 0.6390614273071289, 'epoch': 5.0})

In [27]:
def translate(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(**inputs)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(translate("Hola"))

Hiya.
